In [1]:
import pandas as pd
import os

# 1. Cari alamat folder input secara otomatis
input_dir = ''
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train.csv':
            input_dir = os.path.join(dirname, filename)
            break

# 2. Coba baca datanya menggunakan alamat yang sudah ketemu
try:
    train_df = pd.read_csv(input_dir)
    print(f"Berhasil! File ditemukan di: {input_dir}")
    print(f"Total data yang dimuat: {len(train_df)} baris")
    display(train_df.head()) # Menampilkan tabel dengan cantik
except Exception as e:
    print(f"Masih belum ketemu nih. Errornya: {e}")
    # Jika gagal, tampilkan isi folder input supaya kita bisa cek namanya
    print("\nIsi folder /kaggle/input Anda adalah:")
    for dirname, _, filenames in os.walk('/kaggle/input'):
        print(dirname)

Berhasil! File ditemukan di: /kaggle/input/competitions/llm-classification-finetuning/train.csv
Total data yang dimuat: 57477 baris


,id,model_a,model_b,prompt,response_a,response_b,winner_model_a,winner_model_b,winner_tie
0,30192,gpt-4-1106-preview,gpt-4-0613,"[""Is it morally right to try to have a certain...","[""The question of whether it is morally right ...","[""As an AI, I don't have personal beliefs or o...",1,0,0
1,53567,koala-13b,gpt-4-0613,"[""What is the difference between marriage lice...","[""A marriage license is a legal document that ...","[""A marriage license and a marriage certificat...",0,1,0
2,65089,gpt-3.5-turbo-0613,mistral-medium,"[""explain function calling. how would you call...","[""Function calling is the process of invoking ...","[""Function calling is the process of invoking ...",0,0,1
3,96401,llama-2-13b-chat,mistral-7b-instruct,"[""How can I create a test set for a very rare ...","[""Creating a test set for a very rare category...","[""When building a classifier for a very rare c...",1,0,0
4,198779,koala-13b,gpt-3.5-turbo-0314,"[""What is the best way to travel from Tel-Aviv...","[""The best way to travel from Tel Aviv to Jeru...","[""The best way to travel from Tel-Aviv to Jeru...",0,1,0


In [2]:
import pandas as pd
import numpy as np
import ast
import os

# 1. Fungsi pembersihan teks
def clean_text(text):
    try:
        actual_list = ast.literal_eval(text)
        return " ".join(actual_list)
    except:
        return str(text)

# 2. Cari lokasi file secara otomatis (agar tidak FileNotFoundError lagi)
path_train = ""
path_test = ""
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename == 'train.csv':
            path_train = os.path.join(dirname, filename)
        if filename == 'test.csv':
            path_test = os.path.join(dirname, filename)

# 3. Baca data
train_df = pd.read_csv(path_train)
test_df = pd.read_csv(path_test)

# 4. Bersihkan data TEST (ini yang akan kita kumpulkan)
test_df['response_a_clean'] = test_df['response_a'].apply(clean_text)
test_df['response_b_clean'] = test_df['response_b'].apply(clean_text)

# 5. Strategi Prediksi: Siapa yang lebih panjang jawabannya, dia yang menang
def predict_logic(row):
    len_a = len(row['response_a_clean'].split())
    len_b = len(row['response_b_clean'].split())
    
    if len_a > len_b:
        return [0.5, 0.3, 0.2] # A lebih panjang -> Peluang A besar
    elif len_b > len_a:
        return [0.3, 0.5, 0.2] # B lebih panjang -> Peluang B besar
    else:
        return [0.33, 0.33, 0.34] # Sama panjang -> Seri

# 6. Jalankan prediksi
predictions = test_df.apply(predict_logic, axis=1)
predictions_df = pd.DataFrame(predictions.tolist(), columns=['winner_model_a', 'winner_model_b', 'winner_tie'])

# 7. Buat file submission.csv
submission = pd.concat([test_df['id'], predictions_df], axis=1)
submission.to_csv('submission.csv', index=False)

print("Selesai! File submission.csv sudah siap.")
display(submission.head())

Selesai! File submission.csv sudah siap.


,id,winner_model_a,winner_model_b,winner_tie
0,136060,0.3,0.5,0.2
1,211333,0.5,0.3,0.2
2,1233961,0.5,0.3,0.2
